# Lab 3 — Resampling Lab

**Day 06 · Anomaly Detection · Cisco AI/ML Training**

---

## Learning objectives

1. **Oversample** the minority fraud class on the training set only.
2. Train `LogisticRegression` with and without resampling.
3. Compare **F1 (fraud)** on the same held-out test set.
4. Discuss overfitting risk when duplicating only **8** fraud rows.

> **Checkpoints:** train fraud **8 → 792** · F1 ≈ **0.67** (both approaches on this seed)

**Companion script:** `../scripts/lab03_resampling_lab.py`

## Resampling strategies

| Strategy | This lab | Effect |
|----------|----------|--------|
| **Oversample minority** | `resample(..., replace=True)` | Duplicate fraud rows to match legit count |
| **Undersample majority** | Optional extension | Drop legit rows — loses data |
| **class_weight** | Labs 5–6 | Penalize missing frauds in loss function |

**Critical rule:** split train/test **first**, then resample **train only** — never touch the test set.

---

## 1. Load data and split

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils import resample

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif GH_ROOT.name == "day-06":
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "data" / "credit-card" / "credit_card_transactions.csv").is_file():
            GH_ROOT = parent
            break

NUMERIC_FEATURES = ["amount", "distance_from_home"]
CATEGORICAL_FEATURES = ["merchant_category"]

df = pd.read_csv(GH_ROOT / "data" / "credit-card" / "credit_card_transactions.csv")
X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"train: {len(X_train)} (fraud {int(y_train.sum())})")
print(f"test:  {len(X_test)} (fraud {int(y_test.sum())})")

---

## 2. Oversample fraud in training set

In [ ]:
train_df = X_train.copy()
train_df["is_fraud"] = y_train.values

majority = train_df[train_df["is_fraud"] == 0]
minority = train_df[train_df["is_fraud"] == 1]
minority_up = resample(
    minority,
    replace=True,
    n_samples=len(majority),
    random_state=42,
)
balanced_train = pd.concat([majority, minority_up]).sample(frac=1, random_state=42)

X_bal = balanced_train.drop(columns=["is_fraud"])
y_bal = balanced_train["is_fraud"]

print("Lab 3 — Resampling lab")
print(f"train fraud before: {int(y_train.sum())}, after oversample: {int(y_bal.sum())}")
print(f"balanced train size: {len(X_bal)}")

8 unique fraud rows become **792** with replacement — many duplicates; model may memorize them.

---

## 3. Build preprocessing pipeline

In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ]
)

def make_pipe() -> Pipeline:
    return Pipeline(
        steps=[
            ("preprocess", preprocess),
            ("clf", LogisticRegression(max_iter=1000, random_state=42)),
        ]
    )

---

## 4. Train and compare F1

In [ ]:
pipe_raw = make_pipe()
pipe_raw.fit(X_train, y_train)
f1_raw = f1_score(y_test, pipe_raw.predict(X_test), zero_division=0)

pipe_bal = make_pipe()
pipe_bal.fit(X_bal, y_bal)
f1_bal = f1_score(y_test, pipe_bal.predict(X_test), zero_division=0)

print(f"F1 fraud (no resampling): {f1_raw:.4f}")
print(f"F1 fraud (oversampled train): {f1_bal:.4f}")

compare = pd.DataFrame({
    "approach": ["raw train", "oversampled train"],
    "train_fraud": [int(y_train.sum()), int(y_bal.sum())],
    "F1_fraud": [f1_raw, f1_bal],
})
display(compare.round(4))

On this seed both F1 scores tie at **0.67** — resampling helps training balance but test has only **2** fraud cases.

---

## 5. Optional — undersample majority

In [ ]:
majority_down = resample(
    majority,
    replace=False,
    n_samples=len(minority),
    random_state=42,
)
under_train = pd.concat([majority_down, minority]).sample(frac=1, random_state=42)
X_under = under_train.drop(columns=["is_fraud"])
y_under = under_train["is_fraud"]

pipe_under = make_pipe()
pipe_under.fit(X_under, y_under)
f1_under = f1_score(y_test, pipe_under.predict(X_test), zero_division=0)

print(f"undersampled train size: {len(X_under)} (fraud {int(y_under.sum())})")
print(f"F1 fraud (undersampled train): {f1_under:.4f}")

---

## 6. Checkpoint summary

In [ ]:
assert int(y_train.sum()) == 8
assert int(y_bal.sum()) == 792
assert abs(f1_raw - 0.6667) < 0.05
assert abs(f1_bal - 0.6667) < 0.05
print("✓ All checkpoint assertions passed")

---

## Reflection questions

1. Why resample only after the train/test split?
2. What is the risk of oversampling 8 rows to 792?
3. When might `class_weight='balanced'` be preferable to oversampling?

**Previous:** [Lab 2 — Imbalance analysis](lab02_imbalance_analysis.ipynb)  
**Next:** [Lab 4 — Proximity detector](lab04_proximity_detector.ipynb)